# Full-ARG Step Replay and Traceback

This notebook loads `l25kb_dated_synthetic_full_arg.trees`, builds a tskit node-time reveal `new_rl.ARGTrace`, and lets you jump to any construction step from the present-time initial state to the terminal input graph.


In [10]:
from pathlib import Path
import sys

import numpy as np
import tskit


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "arg").exists() and (path / "new_rl").exists():
            return path
    raise RuntimeError("Could not find project root containing arg/ and new_rl/")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from new_rl import build_trace_from_full_arg

TREE_PATH = PROJECT_ROOT / "arg/validation/output/tsinfer/l25kb_dated_synthetic_full_arg.trees"
TREE_PATH = "/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2_synthetic_full_arg.trees"
TREE_PATH


'/Users/pratik/.lorax/projects/1000Genomes/1kg_chr2_synthetic_full_arg.trees'

In [11]:
ts = tskit.load(str(TREE_PATH))
trace = build_trace_from_full_arg(TREE_PATH)

summary = {
    "sequence_length": ts.sequence_length,
    "trees": ts.num_trees,
    "samples": ts.num_samples,
    "nodes": ts.num_nodes,
    "edges": ts.num_edges,
    "recombination_nodes": sum(1 for node in ts.nodes() if node.flags & 131072),
    "trace_steps": trace.num_steps,
    "trace_events": trace.event_count,
    "recombination_events": trace.recombination_event_count,
    "coalescence_events": trace.coalescence_event_count,
}
summary


{'sequence_length': 243199375.0,
 'trees': 2119205,
 'samples': 5008,
 'nodes': 47668563,
 'edges': 211819125,
 'recombination_nodes': 44922264,
 'trace_steps': 25202423,
 'trace_events': 25202423,
 'recombination_events': 22461132,
 'coalescence_events': 2741291}

In [ ]:
def summarize_state(state, max_lineages=12):
    active = state.active_lineages
    return {
        "step": state.step,
        "current_time": state.current_time,
        "visible_nodes": len(state.visible_node_ids),
        "visible_edges": len(state.visible_edge_ids),
        "active_lineages": len(active),
        "active_lineage_preview": [
            {"node_id": lineage.node_id, "segments": lineage.segments}
            for lineage in active[:max_lineages]
        ],
    }


def event_before_step(step):
    if step <= 0:
        return None
    return trace.event_at_index(step - 1)


def state_at(step):
    state = trace.state_at_step(step)
    return summarize_state(state), event_before_step(step)


initial_state = trace.state_at_step(0)
terminal_state = trace.state_at_step(trace.num_steps)

summarize_state(initial_state), summarize_state(terminal_state)


## Jump to Any Step

Set `STEP` to any integer from `0` to `trace.num_steps`. Step `0` is the present-time sample-only state. Step `trace.num_steps` is the terminal full ARG state.


In [4]:
STEP = 10
state = trace.state_at_step(STEP)
previous_state = trace.previous_state(state) if STEP > 0 else None

result = {
    "selected": summarize_state(state),
    "event_that_created_selected_step": event_before_step(STEP),
    "previous": None if previous_state is None else summarize_state(previous_state),
}
result


{'selected': {'step': 10,
  'current_time': 5828.823204759131,
  'visible_nodes': 23,
  'visible_edges': 27,
  'active_lineages': 15,
  'active_lineage_preview': [{'node_id': 0,
    'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 1, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 2, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 3, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 4, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 5, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 6, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 7, 'segments': ((0.0, 386.0), (23963.0, 25000.0))},
   {'node_id': 13, 'segments': ((386.0, 23963.0),)},
   {'node_id': 22, 'segments': ((386.0, 3440.0),)},
   {'node_id': 24, 'segments': ((3440.0, 9543.0),)},
   {'node_id': 26, 'segments': ((386.0, 3440.0),)}]},
 'event_that_created_selected_step': ARGEvent(step=10, kind='recombination', time=5828.8

## Windowed Graph at Any Step

Use `graph_at_step` for visualization/debugging. For large inputs, always pass a `genomic_range` instead of materializing the whole chromosome graph.


In [5]:
WINDOW = (5000, 7000)
graph = trace.graph_at_step(STEP, genomic_range=WINDOW)

{
    "step": graph["metadata"]["step"],
    "current_time": graph["metadata"]["current_time"],
    "visible_nodes_in_window": len(graph["nodes"]),
    "visible_edges_in_window": len(graph["edges"]),
    "first_nodes": graph["nodes"][:5],
    "first_edges": graph["edges"][:5],
}


{'step': 10,
 'current_time': 5828.823204759131,
 'visible_nodes_in_window': 18,
 'visible_edges_in_window': 15,
 'first_nodes': [{'id': 0,
   'time': 0.0,
   'flags': 1,
   'is_sample': True,
   'is_recombination': False},
  {'id': 1,
   'time': 0.0,
   'flags': 1,
   'is_sample': True,
   'is_recombination': False},
  {'id': 2,
   'time': 0.0,
   'flags': 1,
   'is_sample': True,
   'is_recombination': False},
  {'id': 3,
   'time': 0.0,
   'flags': 1,
   'is_sample': True,
   'is_recombination': False},
  {'id': 4,
   'time': 0.0,
   'flags': 1,
   'is_sample': True,
   'is_recombination': False}],
 'first_edges': [{'id': 0,
   'source': 10,
   'target': 3,
   'left': 5000.0,
   'right': 7000.0},
  {'id': 1, 'source': 10, 'target': 4, 'left': 5000.0, 'right': 7000.0},
  {'id': 2, 'source': 20, 'target': 0, 'left': 5000.0, 'right': 7000.0},
  {'id': 5, 'source': 23, 'target': 2, 'left': 5000.0, 'right': 7000.0},
  {'id': 7, 'source': 24, 'target': 23, 'left': 5000.0, 'right': 7000.0}

In [6]:
# Optional: materialize a debug TreeSequence snapshot for a selected window/step.
# This is intended for small windows, not whole-chromosome interactive use.
partial_ts = trace.to_tree_sequence_at_step(STEP, genomic_range=WINDOW)
{
    "partial_trees": partial_ts.num_trees,
    "partial_nodes": partial_ts.num_nodes,
    "partial_edges": partial_ts.num_edges,
}


{'partial_trees': 4, 'partial_nodes': 46, 'partial_edges': 15}

## Optional Slider

If `ipywidgets` is available in your notebook environment, this slider lets you inspect replay states interactively.


In [7]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=trace.num_steps,
        step=1,
        description="Step",
        continuous_update=False,
    )

    def inspect_step(step):
        state = trace.state_at_step(step)
        return summarize_state(state)

    display(widgets.interact(inspect_step, step=slider))
except ImportError:
    print("ipywidgets is not installed; use STEP manually in the cells above.")


interactive(children=(IntSlider(value=0, continuous_update=False, description='Step', max=25), Output()), _dom…

<function __main__.inspect_step(step)>

In [8]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from matplotlib.lines import Line2D

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


def plot_trace_step(step, genomic_range=None, ax=None, show_labels=True):
    graph = trace.graph_at_step(step, genomic_range=genomic_range)
    state = trace.state_at_step(step)
    event = event_before_step(step)
    nodes = graph["nodes"]
    edges = graph["edges"]

    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5.5))

    node_by_id = {node["id"]: node for node in nodes}
    children = {node["id"]: [] for node in nodes}
    for edge in edges:
        if edge["source"] in children and edge["target"] in node_by_id:
            children[edge["source"]].append(edge["target"])

    sample_ids = sorted(node["id"] for node in nodes if node["is_sample"])
    x_cache = {node_id: float(i) for i, node_id in enumerate(sample_ids)}

    def node_x(node_id, depth=0):
        if node_id in x_cache:
            return x_cache[node_id]
        if depth > len(nodes):
            x_cache[node_id] = float(node_id)
            return x_cache[node_id]
        kids = children.get(node_id, [])
        if not kids:
            x_cache[node_id] = float(len(sample_ids) + (node_id % 7) * 0.15)
        else:
            x_cache[node_id] = float(np.mean([node_x(child, depth + 1) for child in kids]))
        return x_cache[node_id]

    for node in nodes:
        node_x(node["id"])

    active_ids = {lineage.node_id for lineage in state.active_lineages}
    event_ids = set(event.node_ids) if event is not None else set()

    for edge in edges:
        source = node_by_id.get(edge["source"])
        target = node_by_id.get(edge["target"])
        if source is None or target is None:
            continue
        highlight = edge["source"] in event_ids or edge["target"] in event_ids
        ax.plot(
            [x_cache[source["id"]], x_cache[target["id"]]],
            [source["time"], target["time"]],
            color="#d95f02" if highlight else "#c7d0da",
            linewidth=1.8 if highlight else 0.9,
            alpha=0.95 if highlight else 0.75,
            zorder=2 if highlight else 1,
        )

    for node in nodes:
        nid = node["id"]
        if node["is_recombination"]:
            color, marker, size = "#d95f02", "s", 70
        elif node["is_sample"]:
            color, marker, size = "#14E2A8", "o", 70
        else:
            color, marker, size = "#8fa1b3", "o", 55

        if nid in event_ids:
            size += 35
        edgecolor = "#03303E"
        linewidth = 1.4 if nid in event_ids else 0.7
        if nid in active_ids and nid not in event_ids:
            edgecolor = "#2a6fdb"
            linewidth = 1.3

        ax.scatter(
            [x_cache[nid]],
            [node["time"]],
            s=size,
            marker=marker,
            color=color,
            edgecolor=edgecolor,
            linewidth=linewidth,
            zorder=3,
        )
        if show_labels and (
            node["is_sample"]
            or node["is_recombination"]
            or nid in event_ids
            or len(nodes) <= 40
        ):
            ax.text(
                x_cache[nid],
                node["time"],
                str(nid),
                ha="center",
                va="bottom",
                fontsize=7,
                zorder=4,
            )

    max_time = max((node["time"] for node in nodes), default=0.0)
    max_time = max(max_time, float(state.current_time))
    y_pad = max(max_time * 0.08, 1.0)
    ax.axhline(
        state.current_time,
        color="#2a6fdb",
        linestyle="--",
        linewidth=1.0,
        alpha=0.7,
        zorder=0,
    )

    window_txt = (
        "full sequence"
        if genomic_range is None
        else f"window {genomic_range[0]:.0f}-{genomic_range[1]:.0f}"
    )
    event_txt = (
        "initial state"
        if event is None
        else f"{event.kind} @ t={event.time:.2f} nodes={event.node_ids}"
    )
    ax.set_title(
        f"Step {step}/{trace.num_steps} | t={state.current_time:.2f} | "
        f"{len(nodes)} nodes, {len(edges)} edges | {window_txt}\n{event_txt}"
    )
    ax.set_xlabel("sample-order barycenter")
    ax.set_ylabel("time")
    ax.set_ylim(-y_pad, max_time + y_pad)
    ax.grid(alpha=0.15)
    ax.spines[["top", "right"]].set_visible(False)

    legend = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#14E2A8",
            markeredgecolor="#03303E",
            markersize=8,
            label="sample",
        ),
        Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor="#d95f02",
            markeredgecolor="#03303E",
            markersize=8,
            label="recombination",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#8fa1b3",
            markeredgecolor="#03303E",
            markersize=8,
            label="other",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#8fa1b3",
            markeredgecolor="#2a6fdb",
            markersize=8,
            label="active lineage",
        ),
        Line2D([0], [0], color="#2a6fdb", linestyle="--", label="current time"),
    ]
    ax.legend(handles=legend, loc="upper left", frameon=False, fontsize=8)
    return ax, summarize_state(state)



if widgets is None:
    print("ipywidgets is not installed; plotting STEP / WINDOW once.")
    _, summary = plot_trace_step(STEP, genomic_range=WINDOW)
    display(summary)
    plt.show()
else:
    step_slider = widgets.IntSlider(
        value=min(STEP, trace.num_steps),
        min=0,
        max=trace.num_steps,
        step=1,
        description="Step",
        continuous_update=False,
        layout=widgets.Layout(width="60%"),
    )
    use_window = widgets.Checkbox(value=True, description="Use WINDOW")
    left_box = widgets.FloatText(value=float(WINDOW[0]), description="Left")
    right_box = widgets.FloatText(value=float(WINDOW[1]), description="Right")
    out = widgets.Output()

    def render(_=None):
        with out:
            clear_output(wait=True)
            genomic_range = None
            if use_window.value:
                left, right = float(left_box.value), float(right_box.value)
                if right <= left:
                    print("WINDOW requires left < right")
                    return
                genomic_range = (left, right)
            _, summary = plot_trace_step(int(step_slider.value), genomic_range=genomic_range)
            display(summary)
            plt.show()

    for control in (step_slider, use_window, left_box, right_box):
        control.observe(render, names="value")

    display(widgets.VBox([step_slider, widgets.HBox([use_window, left_box, right_box]), out]))
    render()


In [9]:
tss = tskit.load("../arg/validation/output/tsinfer/l25kb_dated_synthetic_full_arg.trees")

In [ ]:
tss.tables.edges